# Kaggle histopathology introduction
This notebook is an introduction to the data challenge of out of distribution classification of histopathology patches. It also serves as a baseline for the code and the model.

If you have any questions, feel free to contact me at [leo.fillioux@centralesupelec.fr](mailto:leo.fillioux@centralesupelec.fr).

In [ ]:
import h5py
import torch
import random
import numpy as np
import pandas as pd
import torchmetrics
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

In [ ]:
TRAIN_IMAGES_PATH = 'train.h5'
VAL_IMAGES_PATH = 'val.h5'
TEST_IMAGES_PATH = 'test.h5'
SEED = 0

In [ ]:
torch.random.manual_seed(SEED)
random.seed(SEED)

## 1. Introduction to the data
The dataset consists of patches of whole slide images which should be classified into either containing tumor or not. The training images come from 3 different centers (i.e. hospitals), while the validation set comes from another center and the test set from yet another center. The visual aspect of the patches are quite different due to the slightly different staining procedures, conditions, and equipment from each hospital. The objective of the task is to build a classifier that is impacted by this distribution shift as little as possible.

The data is stored in `.h5` files, which can be seen as a folder hierarchy, which are can be seen as the following.
```
├── idx           # index of the image
│   └── img       # image in a tensor format
│   └── label     # binary label of the image
│   └── metadata  # some metadata on the images
```
The metadata is included for completeness but is not necessarily useful. The first element in the metadata corresponds to the center.

The following is a visualization of how different the images look from the different centers.

In [ ]:
train_images = {0: {0: None, 1: None},
                3: {0: None, 1: None},
                4: {0: None, 1: None}}
val_images = {1: {0: None, 1: None}}

In [ ]:
for img_data, data_path in zip([train_images, val_images], [TRAIN_IMAGES_PATH, VAL_IMAGES_PATH]):
    with h5py.File(data_path, 'r') as hdf:
        for img_idx in list(hdf.keys()):
            label = int(np.array(hdf.get(img_idx).get('label')))
            center = int(np.array(hdf.get(img_idx).get('metadata'))[0])
            if img_data[center][label] is None:
                img_data[center][label] = np.array(hdf.get(img_idx).get('img'))
            if all(all(value is not None for value in inner_dict.values()) for inner_dict in img_data.values()):
                break
all_data = {**train_images, **val_images}

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(20, 10))
center_ids = {center: idx for idx, center in enumerate(all_data.keys())}
for center in all_data:
    for label in all_data[center]:
        axs[label, center_ids[center]].imshow(np.moveaxis(all_data[center][label], 0, -1).astype(np.float32))
        axs[label, center_ids[center]].axis('off')
        if label == 0:
            axs[label, center_ids[center]].set_title(f'Center {center}')
plt.show()

## 2. Building a baseline model
The baseline model consists of extracting DINOv2 embeddings and linear probing.

In [ ]:
from src.main import get_solution
from src.lib.datasets import BaselineDataset, PrecomputedDataset, precompute

BATCH_SIZE = 16

## 2. Building a baseline model
The baseline now lives in `src/`, but the notebook stays the main place to run it.

In [ ]:
# CONFIG = {
#     'train_path': TRAIN_IMAGES_PATH,
#     'val_path': VAL_IMAGES_PATH,
#     'test_path': TEST_IMAGES_PATH,
#     'output_csv': 'baseline.csv',
#     'batch_size': BATCH_SIZE,
#     'resize': (98, 98),
#     'num_epochs': 100,
#     'patience': 10,
#     'lr': 0.001,
# }

In [ ]:
# solution = get_solution('baseline', CONFIG)

In [ ]:
# history = solution.fit()

In [ ]:
## 3. Making the final prediction

In [ ]:
# solution.predict_test(output_csv='baseline_2.csv')

### Color Jitters

In [ ]:
# COLOR_JITTER_CONFIG = {
#     "train_path": TRAIN_IMAGES_PATH,
#     "val_path": VAL_IMAGES_PATH,
#     "test_path": TEST_IMAGES_PATH,
#     "output_csv": "baseline_color_jitter.csv",
#     "batch_size": BATCH_SIZE,
#     "resize": (98, 98),
#     "num_epochs": 100,
#     "patience": 10,
#     "lr": 0.001,
#     "checkpoint_path": "best_model_color_jitter.pth",
#     "jitter_brightness": 0.15,
#     "jitter_contrast": 0.15,
#     "jitter_saturation": 0.15,
#     "jitter_hue": 0.02,
# }

In [ ]:
# color_jitter_solution = get_solution("baseline_color_jitter", COLOR_JITTER_CONFIG)

In [ ]:
# color_jitter_solution.fit()

In [ ]:
# color_jitter_solution.predict_test(output_csv="baseline_color_jitter.csv")

## Resizing image

In [ ]:
# HIGHRES_CONFIG = {
#     "train_path": TRAIN_IMAGES_PATH,
#     "val_path": VAL_IMAGES_PATH,
#     "test_path": TEST_IMAGES_PATH,
#     "output_csv": "baseline_224.csv",
#     "batch_size": BATCH_SIZE,
#     "resize": (224, 224),
#     "num_epochs": 100,
#     "patience": 10,
#     "lr": 0.001,
#     "checkpoint_path": "best_model_224.pth",
# }

In [ ]:
# highres_solution = get_solution("baseline_224", HIGHRES_CONFIG)

In [ ]:
# highres_solution.fit()

In [ ]:
# highres_solution.predict_test(output_csv="baseline_224.csv")

## Stain Jitters + Higher resolution

In [ ]:
import importlib
import src.lib.augmentations as augmentations_module
import src.lib.datasets as datasets_module
import src.lib.solutions as solutions_module
import src.main as main_module

importlib.reload(augmentations_module)
importlib.reload(datasets_module)
importlib.reload(solutions_module)
importlib.reload(main_module)

In [ ]:
from src.main import get_solution

In [ ]:
TARGETED_224_CONFIG = {
    "train_path": TRAIN_IMAGES_PATH,
    "val_path": VAL_IMAGES_PATH,
    "test_path": TEST_IMAGES_PATH,
    "output_csv": "baseline_224_targeted_augmentations.csv",
    "batch_size": BATCH_SIZE,
    "resize": (224, 224),
    "num_epochs": 100,
    "patience": 10,
    "head_lr": 0.001,
    "checkpoint_path": "best_model_224_targeted_augmentations.pth",
    "stain_sigma": 0.1,
    "jitter_brightness": 0.15,
    "jitter_contrast": 0.15,
}

In [ ]:
targeted_224_solution = get_solution("baseline_224_targeted_augmentations", TARGETED_224_CONFIG)

In [ ]:
targeted_224_history = targeted_224_solution.fit()

In [ ]:
targeted_224_solution.predict_test(output_csv="baseline_224_targeted_augmentations.csv")

In [ ]:
LORA_TARGETED_CONFIG = {
    "train_path": TRAIN_IMAGES_PATH,
    "val_path": VAL_IMAGES_PATH,
    "test_path": TEST_IMAGES_PATH,
    "batch_size": BATCH_SIZE,
    "num_epochs": 100,
    "patience": 10,
    "head_lr": 0.001,
    "backbone_lr": 4e-4,
    "checkpoint_path": "best_model_lora_dinov2_targeted_augmentations.pth",
    "lora_rank": 8,
    "lora_alpha": 1.0,
    "stain_sigma": 0.1,
    "jitter_brightness": 0.15,
    "jitter_contrast": 0.15,
}

lora_targeted_solution = get_solution("lora_dinov2_targeted_augmentations", LORA_TARGETED_CONFIG)

lora_targeted_history = lora_targeted_solution.fit()

lora_targeted_solution.predict_test(output_csv="lora_dinov2_targeted_augmentations.csv")
